# 📊 BIST-100 HMM+MOM3 Hibrit Tarama — TvDatafeed ile
## Kaynak: TradingView (tvdatafeed) | Timeframe: Haftalık | Strateji: HMM × MOM3

**Veri Kaynağı:** [rongardF/tvdatafeed](https://github.com/rongardF/tvdatafeed) → TradingView OHLCV  
**Timeframe:** `Interval.in_weekly` — HMM için optimum (13-52 bar/yıl, düşük gürültü)  
**Lookback:** 200 haftalık bar (~4 yıl)  
**Strateji:** Walk-Forward WFO → GaussianHMM boğa rejimi × 13-haftalık momentum  
**Kalman:** Linear state-space [seviye, hız] → 13 haftalık fiyat hedefi  

> **Not:** WFO analizinde MOM3 (momentum) 5 hisseden 3'ünde, HMM 1'inde kazandı.  
> En güçlü kombinasyon: HMM boğa rejimi filtresi + MOM13W pozitif sinyal birlikte.


In [ ]:
# Kurulum — Colab / Jupyter için
import subprocess, sys

def install(pkg, extra=""):
    cmd = [sys.executable, "-m", "pip", "install", pkg, "-q"]
    if extra:
        cmd.extend(extra.split())
    subprocess.check_call(cmd)

# tvdatafeed (rongardF GitHub reposu — TradingView verisi)
try:
    from tvDatafeed import TvDatafeed, Interval
    print("tvDatafeed zaten kurulu.")
except ImportError:
    print("tvDatafeed kuruluyor (GitHub)...")
    install("git+https://github.com/rongardF/tvdatafeed.git")
    from tvDatafeed import TvDatafeed, Interval

# hmmlearn — Gaussian HMM
try:
    from hmmlearn.hmm import GaussianHMM
    print("hmmlearn zaten kurulu.")
except ImportError:
    print("hmmlearn kuruluyor...")
    install("hmmlearn")
    from hmmlearn.hmm import GaussianHMM

print("\nTüm kütüphaneler hazır!")


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import time
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.linalg import inv
from hmmlearn.hmm import GaussianHMM
from tvDatafeed import TvDatafeed, Interval
from IPython.display import display, HTML

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

print("Import'lar tamam.")
print(f"tvDatafeed sürümü: {TvDatafeed.__module__.split('.')[0]}")


In [ ]:
# TvDatafeed anonim erişim (giriş yapmadan)
# TradingView hesabınız varsa:
#   tv = TvDatafeed(username="kullanici@email.com", password="sifreniz")
# Hesap olmadan da çalışır ama rate-limit daha düşük:

tv = TvDatafeed()

print("TvDatafeed bağlantısı kuruldu.")
print("UYARI: Anonim modda rate-limit vardır (tarama ~3-5 dk sürer).")
print("TradingView hesabı ile daha hızlı çalışır.")

# Bağlantı testi
try:
    test = tv.get_hist("THYAO", "BIST", interval=Interval.in_weekly, n_bars=5)
    print(f"\nBağlantı testi BAŞARILI — THYAO: {len(test)} bar alındı")
    print(test.tail(3))
except Exception as e:
    print(f"\nBağlantı hatası: {e}")
    print("Çözüm: TradingView hesabı ile giriş yapın veya VPN kullanın.")


In [ ]:
# BIST-100 bileşenleri — Haziran 2026 güncel listesi
# TradingView BIST exchange adı: "BIST"
BIST100 = [
    # Büyük cap (index ağırlığı yüksek)
    "THYAO","GARAN","ASELS","BIMAS","EREGL",
    "KCHOL","AKBNK","TUPRS","FROTO","SISE",
    "HALKB","VAKBN","MGROS","ASTOR","TKFEN",
    "ISCTR","TOASO","CCOLA","ENKAI","SAHOL",
    "YKBNK","TCELL","PETKM","PGSUS","KOZAL",
    "OYAKC","ARCLK","KRDMD","DOHOL","SASA",
    # Orta cap
    "TTKOM","AGHOL","ULKER","AEFES","EKGYO",
    "TAVHL","MAVI","BRSAN","GESAN","CWENE",
    "EUPWR","FENER","MIATK","PATEK","QUAGR",
    "KTLEV","CVKMD","HEKTS","PSGYO","ALARK",
    "TABGD","SARKY","AKFGY","AKFEN","SOKM",
    "ODAS","KONTR","KARSN","GOODY","BERA",
    "LOGO","OTKAR","CIMSA","BOLUC","CEMTS",
    # Tarama listesindeki güçlü hisseler
    "AKSA","EFOR","MPARK","NUHCM","ASUZU",
    "CLEBI","ENJSA","GLYHO","GUBRF","INDES",
    "KOZAA","ZOREN","VESTL","VESBE","VKGYO",
    "YATAS","TRILC","TRGYO","NETAS","AFYON",
]
# Tekrar temizle
BIST100 = list(dict.fromkeys(BIST100))

# XU100 endeks referansı
BENCHMARK = "XU100"
EXCHANGE   = "BIST"
N_BARS     = 200    # 200 hafta ≈ 4 yıl (WFO için yeterli)
INTERVAL   = Interval.in_weekly

print(f"Taranacak hisse sayısı: {len(BIST100)}")
print(f"Exchange: {EXCHANGE} | Timeframe: {INTERVAL} | Lookback: {N_BARS} bar")


In [ ]:
def safe_get(symbol, exchange=EXCHANGE, interval=INTERVAL, n_bars=N_BARS, retries=3):
    """TvDatafeed'den güvenli veri çekme — rate-limit ve hata yönetimi ile"""
    for attempt in range(retries):
        try:
            df = tv.get_hist(symbol=symbol, exchange=exchange,
                             interval=interval, n_bars=n_bars)
            if df is not None and len(df) >= 30:
                df.columns = [c.lower() for c in df.columns]
                df.sort_index(inplace=True)
                return df
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    return None

# Benchmark (XU100) indir
print("XU100 endeksi indiriliyor...")
xu100_df = safe_get(BENCHMARK, n_bars=N_BARS)
if xu100_df is not None:
    XU100_WEEKLY = xu100_df["close"].dropna()
    print(f"XU100: {len(XU100_WEEKLY)} bar ({XU100_WEEKLY.index[0].date()} → {XU100_WEEKLY.index[-1].date()})")
else:
    XU100_WEEKLY = None
    print("XU100 indirilemedi — göreceli güç hesaplanamayacak")

# Tüm hisseleri indir
print(f"\n{len(BIST100)} hisse indiriliyor...")
print("(Rate-limit için her hisse arasında kısa bekleme var)\n")

RAW = {}
failed = []

for i, ticker in enumerate(BIST100):
    df = safe_get(ticker)
    if df is not None:
        RAW[ticker] = df
    else:
        failed.append(ticker)
    time.sleep(0.3)   # Rate-limit önlemi
    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(BIST100)}] tamamlandı — başarılı: {len(RAW)}")

print(f"\nİndirme tamamlandı:")
print(f"  Başarılı : {len(RAW)} hisse")
print(f"  Başarısız: {len(failed)} hisse {failed if failed else ''}")


In [ ]:
SPLIT_THRESH = -0.45   # Haftalık -%45'ten büyük düşüş → split, NaN yap

def prepare(ticker, df_raw, xu100_series=None):
    df = df_raw[["open","high","low","close","volume"]].copy()
    df.sort_index(inplace=True)
    df = df[~df.index.duplicated()]
    df.dropna(subset=["close"], inplace=True)
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df.dropna(subset=["close"], inplace=True)

    # Log getiri + split tespiti
    df["log_ret"] = np.log(df["close"] / df["close"].shift(1))
    df["is_split"] = df["log_ret"] < SPLIT_THRESH
    df.loc[df["is_split"], "log_ret"] = np.nan

    # Momentum: 3W ve 13W (~3 aylık)
    df["mom3w"]  = df["close"].pct_change(3)
    df["mom13w"] = df["close"].pct_change(13)

    # Göreceli güç (hisse − XU100)
    if xu100_series is not None:
        xu_aligned = xu100_series.reindex(df.index, method="ffill")
        xu_log = np.log(xu_aligned / xu_aligned.shift(1))
        df["rel_xu100"] = df["log_ret"] - xu_log
    else:
        df["rel_xu100"] = df["log_ret"] - df["log_ret"].rolling(26).mean()

    # SMA 13W / 26W
    df["sma13"] = df["close"].rolling(13).mean()
    df["sma26"] = df["close"].rolling(26).mean()

    # RSI(14)
    delta = df["close"].diff()
    g = delta.clip(lower=0).rolling(14).mean()
    l = (-delta.clip(upper=0)).rolling(14).mean()
    df["rsi14"] = 100 - 100 / (1 + g / l.clip(lower=1e-9))

    # ATR(14)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - df["close"].shift(1)).abs(),
        (df["low"]  - df["close"].shift(1)).abs(),
    ], axis=1).max(axis=1)
    df["atr14"] = tr.rolling(14).mean()

    # Hacim oranı
    df["vol_ratio"] = df["volume"] / df["volume"].rolling(20).mean().clip(lower=1)

    return df

WEEKLY = {}
for ticker, df_raw in RAW.items():
    try:
        WEEKLY[ticker] = prepare(ticker, df_raw, XU100_WEEKLY)
    except Exception as e:
        pass

print(f"{len(WEEKLY)} hisse için indikatörler hesaplandı.")


In [ ]:
class HMMStrategy:
    """
    2-durumlu GaussianHMM.
    Özellikler: [log_ret, rel_xu100]
    Boğa durumu: ortalama log_ret en yüksek durum.
    """
    def __init__(self, n_iter=300, min_samples=30):
        self.n_iter = n_iter
        self.min_samples = min_samples
        self.model = None
        self.bull_state = 0

    def _X(self, df):
        feat = df[["log_ret","rel_xu100"]].replace([np.inf,-np.inf], np.nan).dropna()
        return feat

    def fit(self, df):
        X = self._X(df)
        if len(X) < self.min_samples:
            return False
        try:
            m = GaussianHMM(n_components=2, covariance_type="full",
                            n_iter=self.n_iter, random_state=42)
            m.fit(X.values)
            self.model = m
            self.bull_state = int(np.argmax(m.means_[:, 0]))
            return True
        except:
            return False

    def predict(self, df):
        if self.model is None:
            return pd.Series(0, index=df.index)
        X = self._X(df)
        if X.empty:
            return pd.Series(0, index=df.index)
        try:
            states = self.model.predict(X.values)
            sig = pd.Series((states == self.bull_state).astype(float), index=X.index)
            return sig.reindex(df.index, fill_value=0)
        except:
            return pd.Series(0, index=df.index)

    def bull_prob(self, df):
        """Son bar boğa olasılığı"""
        if self.model is None:
            return 0.5
        X = self._X(df)
        if X.empty:
            return 0.5
        try:
            proba = self.model.predict_proba(X.values)
            return float(proba[-1, self.bull_state])
        except:
            return 0.5

print("HMMStrategy sınıfı hazır.")


In [ ]:
def wfo_hmm(df, train_min=52, test_size=13, step=13):
    """
    Haftalık Walk-Forward:
      train_min = 52 hafta (1 yıl min. eğitim)
      test_size = 13 hafta (~3 ay değerlendirme)
      step      = 13 hafta (kayan pencere adımı)
    """
    n = len(df)
    oos_r, oos_d = [], []
    for fs in range(train_min, n - test_size + 1, step):
        hmm = HMMStrategy()
        if not hmm.fit(df.iloc[:fs]):
            continue
        sig = hmm.predict(df.iloc[:fs+test_size])
        sig_t = sig.iloc[fs:fs+test_size].shift(1).fillna(0)
        ret_t = df["log_ret"].iloc[fs:fs+test_size].fillna(0)
        oos_r.extend((sig_t * ret_t).tolist())
        oos_d.extend(ret_t.index.tolist())

    if not oos_r:
        return {"sharpe": -99, "ret": -99, "n_folds": 0}

    s = pd.Series(oos_r, index=oos_d).sort_index()
    sharpe = float(s.mean()/(s.std()+1e-9)) * np.sqrt(52)
    ret    = float(np.exp(s.sum())-1) * 100
    n_folds = (n - train_min - test_size) // step + 1
    return {"sharpe": round(sharpe,3), "ret": round(ret,1), "n_folds": n_folds}

print("WFO motoru hazır.")


In [ ]:
def kalman_proj(close_series, n_fw=13):
    """
    Lineer Kalman Filtresi (2D: seviye + hız)
    n_fw: kaç hafta ileriye projeksiyon
    Döndürür: (yıllık_hız_pct, hedef_fiyat, alt_bant, üst_bant)
    """
    y = np.log(close_series.dropna().values)
    if len(y) < 20:
        p = float(close_series.iloc[-1])
        return 0, p, p, p

    F = np.array([[1,1],[0,1]])
    H = np.array([[1,0]])
    Q = np.eye(2) * 1e-4
    R = np.array([[1e-2]])
    x = np.array([[y[0]],[0.0]])
    P = np.eye(2)

    for obs in y:
        x_p = F @ x
        P_p = F @ P @ F.T + Q
        inn = obs - (H @ x_p)[0,0]
        S   = H @ P_p @ H.T + R
        K   = P_p @ H.T @ inv(S)
        x   = x_p + K * inn
        P   = (np.eye(2) - K @ H) @ P_p

    vel = float(x[1,0])
    lev = float(x[0,0])
    fut_log = lev + vel * n_fw
    tgt = float(np.exp(fut_log))
    sig = float(np.std(np.diff(y)) * np.sqrt(n_fw))
    return (
        round(vel * 52 * 100, 2),           # Yıllık hız %
        round(tgt, 2),                       # Hedef fiyat
        round(float(np.exp(fut_log-2*sig)),2),  # Alt bant
        round(float(np.exp(fut_log+2*sig)),2),  # Üst bant
    )

print("Kalman projeksiyon modeli hazır.")


In [ ]:
def scan(ticker):
    if ticker not in WEEKLY:
        return None
    df = WEEKLY[ticker].dropna(subset=["log_ret","rel_xu100"])
    if len(df) < 60:
        return None

    cur  = float(df["close"].iloc[-1])
    rsi  = float(df["rsi14"].iloc[-1]) if not pd.isna(df["rsi14"].iloc[-1]) else 50
    atr  = float(df["atr14"].iloc[-1]) if not pd.isna(df["atr14"].iloc[-1]) else cur*0.03
    vol  = float(df["vol_ratio"].iloc[-1]) if not pd.isna(df["vol_ratio"].iloc[-1]) else 1.0
    mom  = float(df["mom13w"].iloc[-1]) if not pd.isna(df["mom13w"].iloc[-1]) else 0
    sma13= float(df["sma13"].iloc[-1]) if not pd.isna(df["sma13"].iloc[-1]) else cur
    sma26= float(df["sma26"].iloc[-1]) if not pd.isna(df["sma26"].iloc[-1]) else cur

    # WFO
    wfo = wfo_hmm(df)

    # Güncel HMM
    hmm = HMMStrategy()
    hmm.fit(df)
    prob = hmm.bull_prob(df)
    hmm_sig = 1 if prob > 0.55 else 0

    # MOM13W sinyali
    mom_sig = 1 if mom > 0 else 0

    # MA trend filtresi
    ma_bull = 1 if cur > sma13 > sma26 else 0

    # Kalman
    vel_pct, tgt, tgt_lo, tgt_hi = kalman_proj(df["close"])
    upside = (tgt/cur - 1)*100

    # Kompozit skor
    # HMM %40 + Kalman hız %25 + MOM %20 + MA trend %10 + Hacim %5
    s_hmm = prob * 100
    s_kal = min(max(vel_pct, 0), 100)
    s_mom = min(max(mom*200, 0), 100)
    s_ma  = ma_bull * 100
    s_vol = min(vol*50, 100)
    composite = 0.40*s_hmm + 0.25*s_kal + 0.20*s_mom + 0.10*s_ma + 0.05*s_vol

    # Hibrit sinyal
    if hmm_sig and mom_sig and ma_bull:
        sinyal = "GÜÇLÜ AL"
    elif hmm_sig and mom_sig:
        sinyal = "AL"
    elif hmm_sig:
        sinyal = "HMM AL"
    elif mom_sig:
        sinyal = "MOM AL"
    else:
        sinyal = "BEKLE"

    return dict(
        ticker=ticker, fiyat=cur, sinyal=sinyal,
        hmm_prob=round(prob*100,1), wfo_sharpe=wfo["sharpe"],
        wfo_ret=wfo["ret"], mom13w=round(mom*100,1),
        kal_vel=vel_pct, kal_hedef=tgt, kal_upside=round(upside,1),
        rsi=round(rsi,1), vol_ratio=round(vol,2),
        stop=round(cur - 2*atr, 2), composite=round(composite,1),
    )

print("Tarama fonksiyonu hazır.")


In [ ]:
print("BIST-100 HMM+MOM3 taraması başlıyor...")
print("(HMM WFO hesabı ~2-4 dakika)\n")

results = []
errs = []

for i, t in enumerate(WEEKLY.keys()):
    try:
        r = scan(t)
        if r:
            results.append(r)
    except Exception as e:
        errs.append((t, str(e)))
    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(WEEKLY)}] tamamlandı...")

print(f"\nTarama bitti: {len(results)} hisse")

DF = pd.DataFrame(results).sort_values("composite", ascending=False)
DF.reset_index(drop=True, inplace=True)
DF.index += 1

print(f"\n{'='*90}")
print(f"BIST-100 HMM+MOM3 HİBRİT TARAMA — {pd.Timestamp.today().strftime('%Y-%m-%d')}")
print(f"{'='*90}")


In [ ]:
COLS = ["ticker","fiyat","sinyal","hmm_prob","mom13w",
        "kal_hedef","kal_upside","kal_vel","wfo_sharpe","rsi","stop","composite"]

# GÜÇLÜ AL
guclu = DF[DF["sinyal"]=="GÜÇLÜ AL"]
print(f"\n🟢 GÜÇLÜ AL ({len(guclu)} hisse): HMM boğa + MOM13W+ + SMA boğa trendi")
print("-"*90)
if len(guclu): print(guclu[COLS].to_string())

# AL
al = DF[DF["sinyal"]=="AL"]
print(f"\n🟡 AL ({len(al)} hisse): HMM boğa + MOM13W+")
print("-"*90)
if len(al): print(al.head(15)[COLS].to_string())

# HMM AL
hmmal = DF[DF["sinyal"]=="HMM AL"]
print(f"\n🔵 HMM AL ({len(hmmal)} hisse): Sadece HMM boğa rejimine girmiş")
print("-"*90)
if len(hmmal): print(hmmal.head(10)[COLS].to_string())

# Top 30 sıralama
print(f"\n{'='*90}")
print("📊 GENEL SIRA (İlk 30 — Kompozit Skoruna göre)")
print("-"*90)
print(DF.head(30)[COLS].to_string())


In [ ]:
top = DF[DF["sinyal"].isin(["GÜÇLÜ AL","AL"])].head(16)
if len(top) == 0:
    top = DF.head(16)

n = len(top)
cols_g = 4
rows_g = (n + cols_g - 1) // cols_g

fig, axes = plt.subplots(rows_g, cols_g, figsize=(22, rows_g*4))
axes = axes.flatten()
fig.suptitle(f"BIST-100 HMM+MOM3 — En Güçlü {n} Sinyal\nHaftalık Fiyat + HMM Rejimleri (Yeşil=Boğa, Kırmızı=Ayı)",
             fontsize=13, fontweight="bold")

hmm_cache = {}

for idx, (_, row) in enumerate(top.iterrows()):
    ax = axes[idx]
    tk = row["ticker"]
    if tk not in WEEKLY:
        ax.set_visible(False)
        continue

    df = WEEKLY[tk].tail(100)

    # HMM durumları
    if tk not in hmm_cache:
        h = HMMStrategy()
        h.fit(WEEKLY[tk])
        hmm_cache[tk] = h
    states = hmm_cache[tk].predict(WEEKLY[tk]).reindex(df.index).fillna(0)

    ax.plot(df.index, df["close"], color="black", lw=1.3, zorder=5)

    # Rejim arka planı
    for j in range(len(df)-1):
        c = "#d4f7d4" if states.iloc[j] == 1 else "#fad4d4"
        ax.axvspan(df.index[j], df.index[j+1], alpha=0.35, color=c, zorder=1)

    # SMA'lar
    ax.plot(df.index, df["sma13"], color="darkorange", lw=0.8, alpha=0.85, label="SMA13W")
    ax.plot(df.index, df["sma26"], color="purple",     lw=0.8, alpha=0.85, label="SMA26W")

    # Kalman hedef
    ax.axhline(row["kal_hedef"], color="blue", ls="--", lw=0.9, alpha=0.8)
    # Stop loss
    ax.axhline(row["stop"], color="crimson", ls=":", lw=0.8, alpha=0.8)

    # Başlık rengi sinyale göre
    tc = {"GÜÇLÜ AL":"darkgreen","AL":"darkorange","HMM AL":"steelblue"}.get(row["sinyal"],"gray")
    ax.set_title(
        f"{tk} [{row['sinyal']}]\n"
        f"P:{row['fiyat']:.0f}  Skor:{row['composite']:.0f}  "
        f"HMM:{row['hmm_prob']:.0f}%  Kal:{row['kal_hedef']:.0f}({row['kal_upside']:+.0f}%)",
        fontsize=8, color=tc, fontweight="bold"
    )
    ax.tick_params(labelsize=6.5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

for i in range(idx+1, len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.savefig("bist_hmm_sinyal.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafik kaydedildi: bist_hmm_sinyal.png")


In [ ]:
# Quarter-Kelly pozisyon boyutlandırma — Sadece AL sinyalleri için
print("\n" + "="*70)
print("💼 QUARTER-KELLY POZİSYON ÖNERİLERİ")
print("   (Güçlü AL + AL sinyalleri)")
print("="*70)

buy_df = DF[DF["sinyal"].isin(["GÜÇLÜ AL","AL"])].head(10).copy()

if len(buy_df) == 0:
    print("Şu an AL sinyali yok.")
else:
    pos_list = []
    for _, r in buy_df.iterrows():
        p = r["hmm_prob"] / 100
        b = max(r["kal_upside"] / 100, 0.05)
        risk = max((r["fiyat"] - r["stop"]) / r["fiyat"], 0.01)
        kelly = max((p*b - (1-p)*risk) / b, 0)
        qk    = min(kelly * 0.25, 0.15)

        pos_list.append({
            "Hisse":      r["ticker"],
            "Sinyal":     r["sinyal"],
            "Fiyat":      r["fiyat"],
            "Hedef":      r["kal_hedef"],
            "Stop":       r["stop"],
            "Risk%":      round(risk*100, 1),
            "Ödül%":      round(b*100, 1),
            "R:R":        round(b/risk, 2) if risk > 0 else 0,
            "Pozisyon%":  round(qk*100, 1),
            "Skor":       r["composite"],
        })

    pos_df = pd.DataFrame(pos_list).sort_values("Pozisyon%", ascending=False)
    pos_df.reset_index(drop=True, inplace=True)
    pos_df.index += 1

    total = pos_df["Pozisyon%"].sum()
    print(pos_df.to_string())
    print("-"*70)
    print(f"Toplam yatırım  : %{total:.1f}")
    print(f"Nakit / tahvil  : %{100-total:.1f}")
    print("\n⚠️  Bu çıktı yatırım tavsiyesi değildir. Kendi analizinizi yapınız.")
